# P3: paired layer4 comparison

Select **Runtime > Change runtime type > GPU**, then run these cells in order. This notebook trains six models (two variants x seeds 0, 1, 2), selects an architecture on dev, and saves evidence to Drive. The final test split is never evaluated. Training may take time; output is printed each epoch.

Use the repository revision containing these P3 files. If the changes have not been pushed, upload `run_comparison.py`, `summarize.py`, and `review_errors.py` when the upload cell opens. No GitHub credentials or push operations are needed.


In [ ]:
import pathlib, subprocess, sys
import torch
assert torch.cuda.is_available(), 'Select a GPU runtime before proceeding'
print(torch.cuda.get_device_name())
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'pandas', 'scikit-learn', 'pillow', 'matplotlib', 'gdown'], check=True)
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
repo = pathlib.Path('/content/road-damage-classification')
if not repo.exists():
    subprocess.run(['git', 'clone', 'https://github.com/laasya-b/road-damage-classification.git', str(repo)], check=True)
# For a local-only P3 change, upload the three P3 scripts from your machine.
if not (repo / 'P3/run_comparison.py').exists():
    from google.colab import files
    uploaded = files.upload()
    for name in ['run_comparison.py', 'summarize.py', 'review_errors.py']:
        if name not in uploaded:
            raise ValueError(f'Upload {name} from the local P3 folder')
        (repo / 'P3' / name).write_bytes(uploaded[name])
print('Repository:', repo)


In [ ]:
import zipfile
import gdown
archive = pathlib.Path('/content/rdd_patches.zip')
patches = pathlib.Path('/content/patches')
if not archive.exists():
    gdown.download(id='17hruEEXaUr4MFu_HD6nwN9BnqPYTB3gk', output=str(archive), quiet=False)
patches.mkdir(exist_ok=True)
with zipfile.ZipFile(archive) as z:
    for member in z.namelist():
        if not (patches / member).resolve().is_relative_to(patches.resolve()):
            raise ValueError('Unsafe ZIP member')
    z.extractall(patches)
# Determine the root that contains paths from P1's CSV, including nested ZIP layouts.
import csv
with (repo / 'P1/splits.csv').open() as f:
    first_path = next(csv.DictReader(f))['path']
candidates = [patches] + [p.parent for p in patches.rglob('Japan') if p.is_dir()]
data_root = next((p for p in candidates if (p / first_path).is_file()), None)
assert data_root is not None, 'Cannot find patch root'
print('Patch root:', data_root)


In [ ]:
runs = pathlib.Path('/content/drive/MyDrive/P3_runs/paired_seeds_0_1_2')
command = [sys.executable, str(repo / 'P3/run_comparison.py'),
           '--data-root', str(data_root), '--out', str(runs)]
if (runs / 'manifest.json').exists():
    command += ['--resume']
# A changed runtime may have changed package versions; in that case use a new runs path.
subprocess.run(command, cwd=repo, check=True)


In [ ]:
import shutil
from IPython.display import Markdown, display, Image
results = repo / 'P3/results'
destination = runs.parent / 'results'
shutil.copytree(results, destination, dirs_exist_ok=True)
display(Markdown((results / 'report.md').read_text()))
display(Image(filename=str(results / 'learning_curves.png')))
display(Image(filename=str(results / 'confusion_matrices.png')))
print('Results saved:', destination)
print('Checkpoints remain in:', runs)
